In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ================== Step 1: Load and Prepare CIFAR-10 Dataset ==================

def load_cifar10_dataset():
    """Load CIFAR-10 dataset"""
    # Define transforms for CIFAR-10
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    # Load train and test datasets
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform)
    
    # CIFAR-10 class names
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck')
    
    return trainset, testset, classes

# ================== Step 2: FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """
    Convert spatial domain images to frequency domain using FFT
    Args:
        images: Tensor of shape (B, C, H, W) in spatial domain
    Returns:
        freq_magnitude: Magnitude spectrum
        freq_phase: Phase spectrum
        freq_complex: Complex frequency representation
    """
    # Apply 2D FFT to each channel
    freq_complex = fft.fft2(images, dim=(-2, -1))
    
    # Shift zero frequency to center
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    # Get magnitude and phase
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # Log scale for better visualization and training
    freq_magnitude_log = torch.log1p(freq_magnitude)
    
    return freq_magnitude_log, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """
    Convert frequency domain back to spatial domain
    Args:
        freq_magnitude: Magnitude spectrum (can be in log scale)
        freq_phase: Phase spectrum
    Returns:
        spatial_images: Reconstructed spatial domain images
    """
    # If magnitude is in log scale, convert back
    freq_magnitude = torch.expm1(freq_magnitude)
    
    # Reconstruct complex frequency representation
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    # Shift back
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    
    # Apply inverse FFT
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    
    # Take real part
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

# ================== Step 3: Create Frequency Domain Dataset ==================

class FrequencyDomainDataset(torch.utils.data.Dataset):
    """Custom dataset that converts images to frequency domain"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        # Get original image and label
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain
        freq_magnitude, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
        
        # Remove batch dimension and combine magnitude and phase as channels
        # Using magnitude for training (you can experiment with both)
        freq_features = freq_magnitude.squeeze(0)
        
        # Store phase for reconstruction
        self.last_phase = freq_phase.squeeze(0)
        
        return freq_features, label, freq_phase.squeeze(0)

# ================== Step 4: Modified CNN Model for Frequency Domain ==================

class FrequencyDomainCNN(nn.Module):
    """Wrapper for pretrained models to work with frequency domain input"""
    
    def __init__(self, base_model='resnet18', num_classes=10):
        super(FrequencyDomainCNN, self).__init__()
        
        # Load pretrained model
        if base_model == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            self.model.fc = nn.Linear(512, num_classes)
        elif base_model == 'vgg16':
            self.model = models.vgg16(pretrained=True)
            self.model.classifier[6] = nn.Linear(4096, num_classes)
        elif base_model == 'mobilenet_v2':
            self.model = models.mobilenet_v2(pretrained=True)
            self.model.classifier[1] = nn.Linear(1280, num_classes)
        else:
            raise ValueError(f"Unsupported model: {base_model}")
        
        # Modify first conv layer if needed (CIFAR-10 images are 32x32)
        if base_model == 'resnet18':
            self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            self.model.maxpool = nn.Identity()
    
    def forward(self, x):
        return self.model(x)

# ================== Step 5: Training Functions ==================

def train_model(model, train_loader, val_loader, epochs=10, lr=0.001):
    """Train the model on frequency domain data"""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    
    train_losses = []
    val_accuracies = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        
        for i, (freq_images, labels, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")):
            freq_images, labels = freq_images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(freq_images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for freq_images, labels, _ in val_loader:
                freq_images, labels = freq_images.to(device), labels.to(device)
                outputs = model(freq_images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_accuracy = 100 * correct / total
        val_accuracies.append(val_accuracy)
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_train_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')
        
        scheduler.step()
    
    return train_losses, val_accuracies

# ================== Step 6: Guided Backpropagation Implementation ==================

class GuidedBackprop:
    """Guided Backpropagation for frequency domain images"""
    
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.gradients = None
        self.forward_relu_outputs = []
        self.hooks = []
        
        # Update ReLU layers for guided backprop
        self._update_relu_layers()
        
        # Register hooks
        self._register_hooks()
    
    def _update_relu_layers(self):
        """Replace ReLU with modified version for guided backprop"""
        
        def relu_backward_hook(module, grad_in, grad_out):
            """Modified ReLU for guided backprop"""
            return (F.relu(grad_in[0]),)
        
        for module in self.model.modules():
            if isinstance(module, nn.ReLU):
                self.hooks.append(module.register_backward_hook(relu_backward_hook))
    
    def _register_hooks(self):
        """Register hooks to capture gradients"""
        
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_in[0]
        
        # Register hook on the first layer
        first_layer = list(self.model.modules())[1]
        self.hooks.append(first_layer.register_backward_hook(backward_hook))
    
    def generate_gradients(self, input_image, target_class):
        """Generate guided gradients for input image"""
        
        # Forward pass
        model_output = self.model(input_image)
        
        # Zero gradients
        self.model.zero_grad()
        
        # Target for backprop
        one_hot = torch.zeros_like(model_output)
        one_hot[0][target_class] = 1.0
        
        # Backward pass
        model_output.backward(gradient=one_hot, retain_graph=True)
        
        # Get gradients
        guided_gradients = input_image.grad.clone()
        
        return guided_gradients
    
    def __del__(self):
        """Remove hooks"""
        for hook in self.hooks:
            hook.remove()

# ================== Step 7: Visualization Functions ==================

def visualize_guided_backprop_frequency_to_spatial(model, freq_image, phase, target_class, original_image):
    """
    Apply guided backprop in frequency domain and map back to spatial domain
    """
    
    # Ensure model is in eval mode
    model.eval()
    
    # Prepare input
    freq_input = freq_image.clone().detach().requires_grad_(True)
    
    # Create guided backprop instance
    gbp = GuidedBackprop(model)
    
    # Generate gradients in frequency domain
    guided_grads_freq = gbp.generate_gradients(freq_input, target_class)
    
    # Process gradients - ensure correct shape
    guided_grads_freq = guided_grads_freq.squeeze(0).cpu().detach()
    
    # Ensure phase has correct shape - should be (3, 32, 32)
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    
    # Convert frequency domain gradients back to spatial domain
    # Use the phase information from the original FFT
    spatial_grads = frequency_to_spatial(guided_grads_freq.unsqueeze(0), phase.unsqueeze(0))
    spatial_grads = spatial_grads.squeeze(0)
    
    # Normalize gradients for visualization
    spatial_grads = torch.abs(spatial_grads)
    
    # Create saliency map (average across channels)
    saliency_map = torch.mean(spatial_grads, dim=0).numpy()
    
    # Normalize saliency map to [0, 1]
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    
    # Get original image for overlay
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    # Prepare original image normalized
    original_np = original_image.cpu().permute(1, 2, 0).numpy()
    original_np = (original_np - original_np.min()) / (original_np.max() - original_np.min() + 1e-8)
    
    # Apply colormap to saliency map and extract RGB
    saliency_colored = plt.cm.hot(saliency_map)[:, :, :3]
    
    # Overlay saliency map on original image
    alpha = 0.5
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    
    # Ensure values are in [0, 1]
    highlighted = np.clip(highlighted, 0, 1)
    
    return spatial_grads, saliency_map, highlighted

def plot_results(original, freq_magnitude, saliency_map, highlighted, prediction, true_label, classes):
    """Plot the complete pipeline results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original image
    if original.dim() == 4:
        original = original.squeeze(0)
    original_np = original.cpu().permute(1, 2, 0).numpy()
    original_np = (original_np - original_np.min()) / (original_np.max() - original_np.min() + 1e-8)
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nTrue: {classes[true_label]}')
    axes[0, 0].axis('off')
    
    # Frequency domain (magnitude spectrum)
    freq_display = freq_magnitude.squeeze(0).mean(0).cpu().numpy()
    axes[0, 1].imshow(freq_display, cmap='gray')
    axes[0, 1].set_title('Frequency Domain\n(Magnitude Spectrum)')
    axes[0, 1].axis('off')
    
    # Model prediction
    axes[0, 2].text(0.5, 0.5, f'Prediction: {classes[prediction]}', 
                    ha='center', va='center', fontsize=16)
    axes[0, 2].set_title('Model Prediction')
    axes[0, 2].axis('off')
    
    # Saliency map
    axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Spatial Domain)')
    axes[1, 0].axis('off')
    
    # Highlighted original
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Regions\n(Important for Prediction)')
    axes[1, 1].axis('off')
    
    # Frequency domain saliency
    axes[1, 2].imshow(saliency_map, cmap='jet')
    axes[1, 2].set_title('Importance Heatmap')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# ================== Step 8: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Frequency Domain CNN with Guided Backpropagation Pipeline")
    print("="*80)
    
    # Step 1: Load CIFAR-10 dataset
    print("\n[Step 1] Loading CIFAR-10 dataset...")
    trainset, testset, classes = load_cifar10_dataset()
    
    # Step 2: Create train/validation split
    print("\n[Step 2] Splitting dataset into train/validation sets...")
    train_indices, val_indices = train_test_split(
        list(range(len(trainset))), 
        test_size=0.2, 
        random_state=42
    )
    
    train_subset = Subset(trainset, train_indices)
    val_subset = Subset(trainset, val_indices)
    
    # Step 3: Create frequency domain datasets
    print("\n[Step 3] Converting to frequency domain...")
    freq_train_dataset = FrequencyDomainDataset(train_subset)
    freq_val_dataset = FrequencyDomainDataset(val_subset)
    freq_test_dataset = FrequencyDomainDataset(testset)
    
    # Create data loaders
    train_loader = DataLoader(freq_train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(freq_val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(freq_test_dataset, batch_size=1, shuffle=False)
    
    # Step 4: Initialize model
    print("\n[Step 4] Initializing ResNet18 model for frequency domain...")
    model = FrequencyDomainCNN(base_model='resnet18', num_classes=10).to(device)
    
    # Step 5: Train model
    print("\n[Step 5] Training model on frequency domain data...")
    print("This will take a few minutes...")
    train_losses, val_accuracies = train_model(
        model, train_loader, val_loader, epochs=5, lr=0.001
    )
    
    # Step 6: Evaluate on test set
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i, (freq_images, labels, _) in enumerate(test_loader):
            if i >= 100:  # Evaluate on first 100 samples
                break
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_accuracy = 100 * correct / total
    print(f"Test Accuracy (first 100 samples): {test_accuracy:.2f}%")
    
    # Step 7: Guided Backpropagation Visualization
    print("\n[Step 7] Applying Guided Backpropagation and visualization...")
    print("Generating explanations for 5 test samples...")
    
    # Get a few test samples for visualization
    for idx in range(5):
        # Get original image
        original_image, true_label = testset[idx]
        
        # Convert to frequency domain
        freq_magnitude, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
        freq_input = freq_magnitude.to(device)
        
        # Get prediction
        model.eval()
        with torch.no_grad():
            output = model(freq_input)
            _, predicted = torch.max(output.data, 1)
            predicted_class = predicted.item()
        
        # Apply guided backpropagation
        freq_input.requires_grad_(True)
        
        # Pass phase without extra unsqueezing (it's already (1, 3, 32, 32))
        spatial_grads, saliency_map, highlighted = visualize_guided_backprop_frequency_to_spatial(
            model, freq_input, phase.squeeze(0), predicted_class, original_image
        )
        
        # Plot results
        print(f"\nSample {idx+1}:")
        print(f"True Label: {classes[true_label]}, Predicted: {classes[predicted_class]}")
        plot_results(
            original_image, 
            freq_magnitude, 
            saliency_map, 
            highlighted,
            predicted_class, 
            true_label, 
            classes
        )
    
    print("\n" + "="*80)
    print("Pipeline completed successfully!")
    print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ================== Step 1: Load and Prepare CIFAR-10 Dataset ==================

def load_cifar10_dataset():
    """Load CIFAR-10 dataset with improved augmentation"""
    # Define transforms for CIFAR-10 with data augmentation
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    # Load train and test datasets
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)
    
    # CIFAR-10 class names
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck')
    
    return trainset, testset, classes

# ================== Step 2: FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """
    Convert spatial domain images to frequency domain using FFT
    Args:
        images: Tensor of shape (B, C, H, W) in spatial domain
    Returns:
        freq_magnitude: Magnitude spectrum
        freq_phase: Phase spectrum
        freq_complex: Complex frequency representation
    """
    # Apply 2D FFT to each channel
    freq_complex = fft.fft2(images, dim=(-2, -1))
    
    # Shift zero frequency to center
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    # Get magnitude and phase
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # Log scale for better visualization and training (with smoothing)
    freq_magnitude_log = torch.log(freq_magnitude + 1e-8)
    
    return freq_magnitude_log, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """
    Convert frequency domain back to spatial domain
    Args:
        freq_magnitude: Magnitude spectrum (can be in log scale)
        freq_phase: Phase spectrum
    Returns:
        spatial_images: Reconstructed spatial domain images
    """
    # If magnitude is in log scale, convert back
    freq_magnitude = torch.exp(freq_magnitude) - 1e-8
    
    # Reconstruct complex frequency representation
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    # Shift back
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    
    # Apply inverse FFT
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    
    # Take real part
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

# ================== Step 3: Create Frequency Domain Dataset ==================

class FrequencyDomainDataset(torch.utils.data.Dataset):
    """Custom dataset that converts images to frequency domain"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        # Get original image and label
        image, label = self.original_dataset[idx]
        
        # Convert to frequency domain
        freq_magnitude, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
        
        # Remove batch dimension
        freq_features = freq_magnitude.squeeze(0)
        
        return freq_features, label, freq_phase.squeeze(0)

# ================== Step 4: Improved CNN Model for Frequency Domain ==================

class FrequencyDomainCNN(nn.Module):
    """Improved wrapper for pretrained models with better initialization"""
    
    def __init__(self, base_model='resnet18', num_classes=10, dropout_rate=0.3):
        super(FrequencyDomainCNN, self).__init__()
        
        # Load pretrained model
        if base_model == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            
            # Modify first conv layer for CIFAR-10 (32x32 images)
            self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            self.model.maxpool = nn.Identity()
            
            # Add dropout for regularization
            self.model.fc = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(512, num_classes)
            )
        elif base_model == 'vgg16':
            self.model = models.vgg16(pretrained=True)
            self.model.classifier[6] = nn.Linear(4096, num_classes)
        elif base_model == 'mobilenet_v2':
            self.model = models.mobilenet_v2(pretrained=True)
            self.model.classifier[1] = nn.Linear(1280, num_classes)
        else:
            raise ValueError(f"Unsupported model: {base_model}")
        
        # Initialize the modified layers properly
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ================== Step 5: Improved Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=7, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=20, lr=0.001, weight_decay=1e-4):
    """Improved training with better optimization strategy"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Use AdamW optimizer with weight decay
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Cosine annealing scheduler for smooth learning rate decay
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6
    )
    
    # Early stopping
    early_stopping = EarlyStopping(patience=10, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(freq_images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            # Update progress bar
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for freq_images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                freq_images, labels = freq_images.to(device), labels.to(device)
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        # Save best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}\n')
        
        scheduler.step()
        
        # Early stopping check
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: FIXED Guided Backpropagation Implementation ==================

class GuidedBackprop:
    """Fixed Guided Backpropagation for frequency domain images"""
    
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.gradients = None
        self.activations = []
        self.handles = []
        
        # Register hooks for guided backpropagation
        self._register_hooks()
    
    def _register_hooks(self):
        """Register forward and backward hooks on ReLU layers"""
        
        def forward_hook(module, input, output):
            """Store activations during forward pass"""
            self.activations.append(output)
        
        def backward_hook(module, grad_input, grad_output):
            """Implement guided backpropagation logic"""
            # Get corresponding activation
            if len(self.activations) > 0:
                activation = self.activations.pop()
                # Only backprop positive gradients through positive activations
                # Create a new tensor to avoid in-place modification
                guided_grad = grad_output[0].clone()
                guided_grad[activation <= 0] = 0
                guided_grad[guided_grad < 0] = 0
                return (guided_grad,)
            return grad_output
        
        # Register hooks on all ReLU layers
        for module in self.model.modules():
            if isinstance(module, nn.ReLU):
                h1 = module.register_forward_hook(forward_hook)
                h2 = module.register_backward_hook(backward_hook)
                self.handles.append(h1)
                self.handles.append(h2)
    
    def generate_gradients(self, input_image, target_class):
        """Generate guided gradients for input image"""
        
        # Reset activations
        self.activations = []
        
        # Ensure input requires grad
        input_image = input_image.clone().detach().requires_grad_(True)
        
        # Forward pass
        model_output = self.model(input_image)
        
        # Zero gradients
        self.model.zero_grad()
        if input_image.grad is not None:
            input_image.grad.zero_()
        
        # Target for backprop
        one_hot = torch.zeros_like(model_output)
        one_hot[0][target_class] = 1.0
        
        # Backward pass
        model_output.backward(gradient=one_hot)
        
        # Get gradients
        guided_gradients = input_image.grad.data.clone()
        
        return guided_gradients
    
    def __del__(self):
        """Remove hooks"""
        for handle in self.handles:
            handle.remove()

# ================== Step 7: Improved Visualization Functions ==================

def visualize_guided_backprop_frequency_to_spatial(model, freq_image, phase, target_class, original_image):
    """
    Apply guided backprop in frequency domain and map back to spatial domain
    with improved visualization
    """
    
    # Ensure model is in eval mode
    model.eval()
    
    # Prepare input
    freq_input = freq_image.clone().detach().to(device)
    
    # Create guided backprop instance
    gbp = GuidedBackprop(model)
    
    # Generate gradients in frequency domain
    guided_grads_freq = gbp.generate_gradients(freq_input, target_class)
    
    # Process gradients
    guided_grads_freq = guided_grads_freq.squeeze(0).cpu().detach()
    
    # Ensure phase has correct shape
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    
    # Convert frequency domain gradients back to spatial domain
    spatial_grads = frequency_to_spatial(guided_grads_freq.unsqueeze(0), phase.unsqueeze(0))
    spatial_grads = spatial_grads.squeeze(0)
    
    # Normalize gradients for visualization
    spatial_grads = torch.abs(spatial_grads)
    
    # Create saliency map (average across channels)
    saliency_map = torch.mean(spatial_grads, dim=0).numpy()
    
    # Apply Gaussian smoothing for better visualization
    try:
        from scipy.ndimage import gaussian_filter
        saliency_map = gaussian_filter(saliency_map, sigma=1.5)
    except ImportError:
        # Fallback if scipy is not available
        pass
    
    # Normalize saliency map to [0, 1]
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    
    # Apply threshold to focus on important regions
    threshold = np.percentile(saliency_map, 60)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)
    
    # Renormalize after thresholding
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    
    # Get original image for overlay
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    # Denormalize original image for better visualization
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()
    
    # Apply colormap to saliency map
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    
    # Improved overlay with better alpha blending
    # Reduce alpha where saliency is low to keep original image more visible
    alpha = 0.6 * saliency_map[:, :, np.newaxis]  # Variable alpha based on saliency
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)
    
    return spatial_grads, saliency_map, highlighted, original_np

def plot_results(original_np, freq_magnitude, saliency_map, highlighted, prediction, true_label, classes, confidence):
    """Plot the complete pipeline results with improved visualization"""
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.suptitle('Frequency Domain CNN - Guided Backpropagation Analysis', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Original image
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}', 
                         fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Frequency domain (magnitude spectrum)
    freq_display = freq_magnitude.squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Magnitude Spectrum)', 
                         fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    # Model prediction with confidence
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 2].text(0.5, 0.5, f'{correct} Prediction: {classes[prediction]}\nConfidence: {confidence:.1f}%', 
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 2].set_title('Model Prediction', fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Saliency map with colorbar
    im2 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Spatial Domain)', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    # Highlighted original (main result)
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Important Regions\n(Overlay on Original)', 
                         fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Heatmap overlay
    im3 = axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50% Overlay)', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 8: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Improved Frequency Domain CNN with Guided Backpropagation Pipeline")
    print("="*80)
    
    # Step 1: Load CIFAR-10 dataset
    print("\n[Step 1] Loading CIFAR-10 dataset...")
    trainset, testset, classes = load_cifar10_dataset()
    
    # Step 2: Create train/validation split
    print("\n[Step 2] Splitting dataset into train/validation sets...")
    train_indices, val_indices = train_test_split(
        list(range(len(trainset))), 
        test_size=0.15,  # Slightly smaller validation set
        random_state=42,
        stratify=[trainset[i][1] for i in range(len(trainset))]  # Stratified split
    )
    
    train_subset = Subset(trainset, train_indices)
    val_subset = Subset(trainset, val_indices)
    
    # Step 3: Create frequency domain datasets
    print("\n[Step 3] Converting to frequency domain...")
    freq_train_dataset = FrequencyDomainDataset(train_subset)
    freq_val_dataset = FrequencyDomainDataset(val_subset)
    freq_test_dataset = FrequencyDomainDataset(testset)
    
    # Create data loaders with optimal batch size
    train_loader = DataLoader(freq_train_dataset, batch_size=128, shuffle=True, 
                             num_workers=2, pin_memory=True)
    val_loader = DataLoader(freq_val_dataset, batch_size=128, shuffle=False,
                           num_workers=2, pin_memory=True)
    test_loader = DataLoader(freq_test_dataset, batch_size=1, shuffle=False)
    
    # Step 4: Initialize model
    print("\n[Step 4] Initializing ResNet18 model for frequency domain...")
    model = FrequencyDomainCNN(base_model='resnet18', num_classes=10, dropout_rate=0.3).to(device)
    
    # Step 5: Train model with improved parameters
    print("\n[Step 5] Training model on frequency domain data...")
    print("Training with improved optimization strategy...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader, 
        epochs=25,  # Increased epochs with early stopping
        lr=0.0005,  # Lower learning rate for stability
        weight_decay=5e-4  # Weight decay for regularization
    )
    
    # Plot training curves
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    # Step 6: Evaluate on test set
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    # Step 7: Guided Backpropagation Visualization
    print("\n[Step 7] Applying Guided Backpropagation and visualization...")
    print("Generating explanations for 5 test samples...")
    
    # Get a few test samples for visualization
    for idx in [0, 10, 20, 30, 40]:  # Diverse samples
        # Get original image
        original_image, true_label = testset[idx]
        
        # Convert to frequency domain
        freq_magnitude, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
        freq_input = freq_magnitude.to(device)
        
        # Get prediction with confidence
        model.eval()
        with torch.no_grad():
            output = model(freq_input)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities.data, 1)
            predicted_class = predicted.item()
            confidence = confidence.item() * 100
        
        # Apply guided backpropagation
        spatial_grads, saliency_map, highlighted, original_np = visualize_guided_backprop_frequency_to_spatial(
            model, freq_input, phase.squeeze(0), predicted_class, original_image
        )
        
        # Plot results
        print(f"\nSample {idx}:")
        print(f"True Label: {classes[true_label]}, Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
        plot_results(
            original_np,
            freq_magnitude, 
            saliency_map, 
            highlighted,
            predicted_class, 
            true_label, 
            classes,
            confidence
        )
    
    print("\n" + "="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("="*80)

if __name__ == "__main__":
    main()

# ***Main Code***

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ================== Step 1: Load and Prepare CIFAR-10 Dataset ==================

def load_cifar10_dataset():
    """Load CIFAR-10 dataset with improved augmentation"""
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)
    
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck')
    
    return trainset, testset, classes

# ================== Step 2: IMPROVED FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """
    IMPROVED: Better frequency domain representation with dual-channel approach
    """
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    freq_magnitude = torch.exp(freq_magnitude)
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

# ================== Step 3: IMPROVED Frequency Domain Dataset ==================

class FrequencyDomainDataset(torch.utils.data.Dataset):
    """IMPROVED: Custom dataset with better frequency features"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
        freq_features = freq_features.squeeze(0)
        freq_phase = freq_phase.squeeze(0)
        
        return freq_features, label, freq_phase

# ================== Step 4: IMPROVED CNN Model for Frequency Domain ==================

def disable_inplace_operations(model):
    """Disable all inplace operations in the model (CRITICAL for Guided Backprop)"""
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
        elif isinstance(module, nn.LeakyReLU):
            module.inplace = False
        elif isinstance(module, nn.ELU):
            module.inplace = False

class FrequencyDomainCNN(nn.Module):
    """IMPROVED: Model architecture optimized for frequency domain"""
    
    def __init__(self, base_model='resnet18', num_classes=10, dropout_rate=0.4):
        super(FrequencyDomainCNN, self).__init__()
        
        if base_model == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            
            # CRITICAL: Disable inplace operations for Guided Backprop
            disable_inplace_operations(self.model)
            
            self.model.conv1 = nn.Conv2d(9, 64, kernel_size=3, stride=1, padding=1, bias=False)
            self.model.maxpool = nn.Identity()
            
            self.model.fc = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(512, 256),
                nn.ReLU(inplace=False),  # CRITICAL: inplace=False
                nn.Dropout(dropout_rate * 0.5),
                nn.Linear(256, num_classes)
            )
        else:
            raise ValueError(f"Unsupported model: {base_model}")
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ================== Step 5: Improved Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=7, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=30, lr=0.001, weight_decay=1e-4):
    """IMPROVED training with better optimization"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr * 10,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(freq_images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for freq_images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                freq_images, labels = freq_images.to(device), labels.to(device)
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}\n')
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
        # Re-disable inplace operations after loading state dict
        disable_inplace_operations(model)
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: FIXED Guided Backpropagation Implementation ==================

class GuidedBackprop:
    """FIXED: Guided Backpropagation for frequency domain images"""
    
    def __init__(self, model):
        self.model = model
        self.model.eval()
        
        # CRITICAL: Ensure all inplace operations are disabled
        disable_inplace_operations(self.model)
        
        self.gradients = None
        self.hooks = []
        
        self._register_hooks()
    
    def _register_hooks(self):
        """Register hooks with proper gradient handling"""
        
        def relu_backward_hook_wrapper(module, grad_in, grad_out):
            """FIXED: Properly handle gradients without in-place modification"""
            if isinstance(grad_in, tuple) and len(grad_in) > 0 and grad_in[0] is not None:
                # Clone to avoid in-place modification
                modified_grad = grad_in[0].clone()
                # Apply guided backprop: only keep positive gradients
                modified_grad = torch.clamp(modified_grad, min=0)
                return (modified_grad,) + grad_in[1:]
            return grad_in
        
        # Register hooks on all ReLU layers
        for name, module in self.model.named_modules():
            if isinstance(module, nn.ReLU):
                hook = module.register_full_backward_hook(relu_backward_hook_wrapper)
                self.hooks.append(hook)
        
        # Hook to capture input gradients
        def input_hook(module, grad_in, grad_out):
            if isinstance(grad_in, tuple) and len(grad_in) > 0 and grad_in[0] is not None:
                self.gradients = grad_in[0].clone()
        
        # Find first conv layer and register hook
        for module in self.model.modules():
            if isinstance(module, nn.Conv2d):
                self.hooks.append(module.register_full_backward_hook(input_hook))
                break
    
    def generate_gradients(self, input_image, target_class):
        """Generate guided gradients for input image"""
        
        # Clone and enable gradients
        input_image = input_image.clone().detach().to(device)
        input_image.requires_grad_(True)
        
        # Forward pass
        model_output = self.model(input_image)
        
        # Zero gradients
        self.model.zero_grad()
        if input_image.grad is not None:
            input_image.grad.zero_()
        
        # Create one-hot target
        one_hot = torch.zeros_like(model_output)
        one_hot[0][target_class] = 1.0
        
        # Backward pass
        model_output.backward(gradient=one_hot, retain_graph=False)
        
        # Get gradients
        guided_gradients = input_image.grad.clone() if input_image.grad is not None else self.gradients
        
        return guided_gradients
    
    def __del__(self):
        """Clean up hooks"""
        for hook in self.hooks:
            hook.remove()

# ================== Step 7: Visualization Functions ==================

def visualize_guided_backprop_frequency_to_spatial(model, freq_image, phase, target_class, original_image):
    """Apply guided backprop and map to spatial domain"""
    
    model.eval()
    freq_input = freq_image.clone().detach().to(device)
    
    gbp = GuidedBackprop(model)
    guided_grads_freq = gbp.generate_gradients(freq_input, target_class)
    
    # Extract only magnitude gradients (first 3 channels)
    guided_grads_mag = guided_grads_freq[:, :3, :, :].squeeze(0).cpu().detach()
    
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    
    spatial_grads = frequency_to_spatial(guided_grads_mag.unsqueeze(0), phase.unsqueeze(0))
    spatial_grads = spatial_grads.squeeze(0)
    
    spatial_grads = torch.abs(spatial_grads)
    saliency_map = torch.mean(spatial_grads, dim=0).numpy()
    
    try:
        from scipy.ndimage import gaussian_filter
        saliency_map = gaussian_filter(saliency_map, sigma=1.5)
    except ImportError:
        pass
    
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    
    threshold = np.percentile(saliency_map, 60)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)
    
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()
    
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha = 0.6 * saliency_map[:, :, np.newaxis]
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)
    
    # Clean up
    del gbp
    
    return spatial_grads, saliency_map, highlighted, original_np

def plot_results(original_np, freq_magnitude, saliency_map, highlighted, prediction, true_label, classes, confidence):
    """Plot the complete pipeline results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.suptitle('Frequency Domain CNN - Guided Backpropagation Analysis', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}', 
                         fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Magnitude Spectrum)', 
                         fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 2].text(0.5, 0.5, f'{correct} Prediction: {classes[prediction]}\nConfidence: {confidence:.1f}%', 
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 2].set_title('Model Prediction', fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')
    
    im2 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(Spatial Domain)', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Important Regions\n(Overlay on Original)', 
                         fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    
    im3 = axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50% Overlay)', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 8: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Improved Frequency Domain CNN with Guided Backpropagation Pipeline")
    print("="*80)
    
    print("\n[Step 1] Loading CIFAR-10 dataset...")
    trainset, testset, classes = load_cifar10_dataset()
    
    print("\n[Step 2] Splitting dataset into train/validation sets...")
    train_indices, val_indices = train_test_split(
        list(range(len(trainset))), 
        test_size=0.15,
        random_state=42,
        stratify=[trainset[i][1] for i in range(len(trainset))]
    )
    
    train_subset = Subset(trainset, train_indices)
    val_subset = Subset(trainset, val_indices)
    
    print("\n[Step 3] Converting to frequency domain...")
    freq_train_dataset = FrequencyDomainDataset(train_subset)
    freq_val_dataset = FrequencyDomainDataset(val_subset)
    freq_test_dataset = FrequencyDomainDataset(testset)
    
    train_loader = DataLoader(freq_train_dataset, batch_size=128, shuffle=True, 
                             num_workers=2, pin_memory=True)
    val_loader = DataLoader(freq_val_dataset, batch_size=128, shuffle=False,
                           num_workers=2, pin_memory=True)
    test_loader = DataLoader(freq_test_dataset, batch_size=1, shuffle=False)
    
    print("\n[Step 4] Initializing ResNet18 model for frequency domain...")
    model = FrequencyDomainCNN(base_model='resnet18', num_classes=10, dropout_rate=0.4).to(device)
    
    print("\n[Step 5] Training model on frequency domain data...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader, 
        epochs=30,
        lr=0.001,
        weight_decay=5e-4
    )
    
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    print("\n[Step 7] Applying Guided Backpropagation and visualization...")
    
    for idx in [0, 10, 20, 30, 40]:
        original_image, true_label = testset[idx]
        
        freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
        freq_input = freq_features.to(device)
        
        model.eval()
        with torch.no_grad():
            output = model(freq_input)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities.data, 1)
            predicted_class = predicted.item()
            confidence = confidence.item() * 100
        
        spatial_grads, saliency_map, highlighted, original_np = visualize_guided_backprop_frequency_to_spatial(
            model, freq_input, phase.squeeze(0), predicted_class, original_image
        )
        
        print(f"\nSample {idx}:")
        print(f"True Label: {classes[true_label]}, Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
        plot_results(
            original_np,
            freq_features, 
            saliency_map, 
            highlighted,
            predicted_class, 
            true_label, 
            classes,
            confidence
        )
        
        # Clear GPU cache
        torch.cuda.empty_cache()
    
    print("\n" + "="*80)
    print("Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("="*80)

if __name__ == "__main__":
    main()